The goal is to:
- Learn patterns between resumes and job descriptions
- Predict candidate suitability
- Compare multiple models and select the best one


In [24]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import os

In [25]:
pip install xgboost

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split 
from sentence_transformers import SentenceTransformer

In [27]:
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 920.74it/s]


In [28]:
X_train = sp.load_npz("../data/X_train.npz")
X_test  = sp.load_npz("../data/X_test.npz")
y_train = np.load("../data/y_train.npy")
y_test  = np.load("../data/y_test.npy")
# Load BERT embeddings
bert_resume_train = np.load("../data/train_resume_embeddings.npy")
bert_jd_train = np.load("../data/train_jd_embeddings.npy")

bert_resume_test = np.load("../data/test_resume_embeddings.npy")
bert_jd_test = np.load("../data/test_jd_embeddings.npy")

print("BERT resume train:", bert_resume_train.shape)
print("BERT JD train:", bert_jd_train.shape)
print("✅ Data loaded!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

BERT resume train: (6240, 384)
BERT JD train: (6240, 384)
✅ Data loaded!
X_train: (6240, 10001)
X_test: (1759, 10001)


In [29]:
# Combine resume + JD embeddings
bert_train = np.hstack([bert_resume_train, bert_jd_train])
bert_test = np.hstack([bert_resume_test, bert_jd_test])

In [30]:
from scipy.sparse import hstack

X_train = hstack([X_train, bert_train])
X_test  = hstack([X_test, bert_test])

print("Final X_train shape:", X_train.shape)

Final X_train shape: (6240, 10769)


In [31]:
## Training Models

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        solver='lbfgs'   # ✅ supports multiclass automatically
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=42,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )
}

The models are trained on:
- TF-IDF vectors of combined resume and job description text
- These vectors represent important keywords in numerical form

In [33]:
results = {}

for name, model in models.items():
    print(f"\n🔹 Training {name}...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')

    results[name] = {
        'model': model,
        'accuracy': acc,
        'f1': f1,
        'y_pred': y_pred
    }

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")


🔹 Training Logistic Regression...
Accuracy: 0.5253
F1 Score: 0.4952

🔹 Training Random Forest...
Accuracy: 0.5321
F1 Score: 0.4275

🔹 Training XGBoost...


c:\Users\diyaa\anaconda3\envs\bert_env\lib\site-packages\xgboost\training.py:200: UserWarning: [22:41:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.5406
F1 Score: 0.5170


In [39]:
print(results.keys())

dict_keys(['Logistic Regression', 'Random Forest', 'XGBoost'])


We trained three different machine learning models on the TF-IDF features extracted from the cleaned dataset:

- Logistic Regression  
- Random Forest  
- XGBoost  

### Training Results

| Model                | Accuracy | F1 Score |
|---------------------|----------|----------|
| Logistic Regression | 0.5253   | 0.4952   |
| Random Forest       | 0.5321   | 0.4275   |
| XGBoost             | 0.5406   | 0.5170   |


### Model Selection

Although Random Forest achieved the highest accuracy, we selected **XGBoost** as the best model because it achieved the highest **F1 Score (0.4899)**.

F1 Score is preferred over accuracy because:
- It balances precision and recall
- It is more reliable for imbalanced datasets
- It ensures better performance across all classes

In [40]:
import joblib
import os

# create folder
os.makedirs("../models", exist_ok=True)

# select best model (you already know XGBoost is best)
best_model = results["XGBoost"]['model']

# save it
joblib.dump(best_model, "../models/best_model.pkl")

print("✅ Best model saved!")

✅ Best model saved!
